# Cross BC Alignment

- Loads the hdf file results from the downstream HEC-RAS model hdf output file
- gets plan information from SST out files
- overwrites BC conditions (external) as a rating curve to align WSELs at boundary
- ideally a breakline would be added along the BC to facilitate cell alignment (not coded)

## DETAILED STEPS
1) Assume that the post auto bc creation files are included in the "outputs folder"
2) Assume that the cloud outputs from the downstream model are included in the inputs/cloud_output folder
3) Identify upstream hucs
4) Identify pair of overlapping junctions (using dictionary files)
5) Identify storm event for each frequency that controls flooding for each pair of junctions
6) Copy relevant output plan DSS file to upstream huc hydrology folder
7) Assign rating curve to upstream hucs based on the stage/flow relationship for the falling limb of the downstream models inflow hydrograph


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#imports
import os
import pathlib as pl

## Set directories and user defined variables

In [3]:
#set working directory and folder variables
os.chdir('..')
home = pl.Path(os.getcwd())

#user to set variables for the project. tagged as parameter for papermill runs
project = 'wy_fy22'
#this is the ds huc of the target huc10
huc = '1404010107'

In [4]:
home = pl.Path(home)
assert home.stem == '_code', 'restart kernel and rerun code'
print('home is at ',home)
from src.hdf import *

inputs = home/'inputs'
outputs_base = home/'outputs'
#create input and outputs folder if not already dirs
if not os.path.exists(inputs):
    os.makedirs(inputs)
if not os.path.exists(outputs_base):
    os.makedirs(outputs)

home is at  U:\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code


In [5]:
#hdf version check
print(h5py.__version__)# Make sure the version is 2.9 or above

3.8.0


In [6]:
#hard variables for project
model_name = f'wy_gdg_{huc}' #folder within inputs folder
junctions = 'shp' #temp
huc_map = {}
force = True #deletes outputs from any prior runs. Set to False to avoid this

## Get Relevant RAS Files

In [7]:
plan_path, in_geom_path, in_flow_path = get_current_ras_files(inputs/project/'cloud_output'/huc/f'{model_name}.prj')

## Get Dictionary files

In [8]:
#get upstream models
with open(inputs/project/'dictionaries'/'HUC10_outflow_toHUC10.json') as src:
    huc_connect_huc = json.load(src)
with open(inputs/project/'dictionaries'/'HUC10_into_dsJunction.json') as src:
    huc_connect_j = json.load(src)
with open(inputs/project/'dictionaries'/'HUC10_Junctions.json') as src:
    huc_js = json.load(src)
with open(inputs/project/'dictionaries'/'junc_res_sink_next_junc_down.json') as src:
    j_to_j = json.load(src)
with open(inputs/project/'dictionaries'/'tie_in_dictionary.json') as src:
    tie_in_events = json.load(src)
with open(inputs/project/'dictionaries'/'rating_or_stage.json') as rc:
    rc_dict = json.load(rc)

## Get HUC BC connections

In [9]:
us_hucs = []
us_j = []
bc_connections = {'junctions':{},'dss_path':{},'ts':{}}
for key, val in huc_connect_huc.items():
    if val == huc:
        us_hucs.append(key)
        us_j.append(huc_connect_j[key])
        bc_connections['junctions'][key] = huc_connect_j[key]
        bc_connections['dss_path'][key] = {}

In [10]:
#this print statement should include your target huc
print(f'Creating new ds boundary contions for {us_hucs}')

Creating new ds boundary contions for ['1404010102', '1404010103', '1404010104', '1404010105', '1404010206']


### Get Recurrence Interval Information for Select HUC

## Copy DSS Files

In [11]:
bc_connections['junctions'].items()

dict_items([('1404010102', 'HUC_101_J_96'), ('1404010103', 'HUC_101_J_69'), ('1404010104', 'HUC_101_J_139'), ('1404010105', 'HUC_101_J_194'), ('1404010206', 'HUC_102_J_207')])

In [12]:
for us_huc, j in  bc_connections['junctions'].items():
    #copy autobc creation files to eb_mod folder for adjustment
    assert os.path.exists(str(outputs_base/project/f'wy_gdg_{us_huc}')), 'Please run auto bc creation tool before running cross bc alignment tool'
    if os.path.exists(str(outputs_base/project/'eb_mod'/f'wy_gdg_{us_huc}')):
        shutil.rmtree(str(outputs_base/project/'eb_mod'/f'wy_gdg_{us_huc}'))
    shutil.copytree(str(outputs_base/project/f'wy_gdg_{us_huc}'),str(outputs_base/project/'eb_mod'/f'wy_gdg_{us_huc}'))
    assert os.path.exists(str(outputs_base/project/'trial_us_to_ds_events'/f'wy_gdg_{us_huc}')), 'Please run trial_us_event_transfer_ds tool before running cross bc alignment tool'
    
    
    #########
    shutil.copytree(str(outputs_base/project/'trial_us_to_ds_events'/f'wy_gdg_{us_huc}'),str(outputs_base/project/'eb_mod'/f'wy_gdg_{us_huc}'), dirs_exist_ok=True)
    #########
    
    
    #create output folder if it doesn't exist
    if not os.path.exists(outputs_base/project/'eb_mod'/f'wy_gdg_{us_huc}'/'Hydrology'):
        os.makedirs(outputs_base/project/'eb_mod'/f'wy_gdg_{us_huc}'/'Hydrology')
    plan_dss_in = str(inputs/project/'cloud_output'/huc/f'{model_name}.dss')
    plan_dss_out = str(outputs_base/project/'eb_mod'/f'wy_gdg_{us_huc}'/'Hydrology'/f'{model_name}.dss')
    shutil.copy(plan_dss_in,plan_dss_out)

In [13]:
bc_connections

{'junctions': {'1404010102': 'HUC_101_J_96',
  '1404010103': 'HUC_101_J_69',
  '1404010104': 'HUC_101_J_139',
  '1404010105': 'HUC_101_J_194',
  '1404010206': 'HUC_102_J_207'},
 'dss_path': {'1404010102': {},
  '1404010103': {},
  '1404010104': {},
  '1404010105': {},
  '1404010206': {}},
 'ts': {}}

# Review and update Upstream HUCs

## Update DSS Files

In [14]:
## get stage hydrograph from us inflow junction and convert to a rating curve and apply it to us_huc downstream BC output file
flow_number_dict = {}
for us_huc, j in  bc_connections['junctions'].items():
    rating_or_stage = rc_dict[us_huc]
    #get downstream bc time series and dss file
    dss_ds = str(outputs_base/project/'eb_mod'/f'wy_gdg_{us_huc}'/'Hydrology'/f'{model_name}.dss')
    fid = HecDss.Open(str(dss_ds))
    pathname_pattern ="/*/*/*/*/*/*/"
    dss_list = fid.getPathnameList(pathname_pattern,sort=1)
    #get upstream dss file and plan number
    us_model_name = f'wy_gdg_{us_huc}' #folder within inputs folder
    ##read plan names and get relevant plan output file to be copied to upstream hucs
    plan_number_dict = {}
    flow_number_dict[us_huc] ={}
    #get plan and flow path info
    plan_path, in_geom_path, in_flow_path = get_current_ras_files(outputs_base/project/'eb_mod'/f'wy_gdg_{us_huc}'/f'{us_model_name}.prj')
    plan_files = glob.glob(str(plan_path.parent/'*.p*[!rj][!.hdf]'))
    for p in plan_files:
        #read old text file
        with open(p, "r+") as f:
            file_contents = f.read()
        plan_title_finder= '(?<=Plan Title=)[\d\w]+(?=_output)'
        f = re.search(plan_title_finder,file_contents)
        flow_finder = '(?<=Plan Title=)[\d\w]+(?=_output)'
        u = re.search(flow_finder,file_contents)
        if f:
            f_num = re.search('.p[\d]+',p)
            plan_number_dict[f.group()] = f_num.group()
            u_num = re.search('u[\d]+',file_contents)
            flow_number_dict[us_huc][f.group()] = u_num.group()
    j_ = list(tie_in_events[us_huc].keys())        
    for ri, event in tie_in_events[us_huc][j_[0]].items():
        event_name = event.replace('-','_')
        event_finder = f'[\d\w\S\s]+{j}/[\d\w\S]+{event_name}_output_[\d\w\S]+'
        #get all relevant dss paths and condense using wildcard
        r = re.compile(event_finder)
        dss_matches_ds = list(filter(r.match, dss_list))

        stage_data_ds = list(filter(re.compile('[\d\w\S\s]+/STAGE/[\d\w\S]+').match, dss_matches_ds))
        flow_data_ds = list(filter(re.compile('[\d\w\S\s]+/FLOW/[\d\w\S]+').match, dss_matches_ds))
        
        #event_us = events_dict_us[us_huc][j][ri].replace('-','_')
        event_us = event.replace('-','_')
        
        #get upstream event and dss_matches based on event
        dss_us = str(outputs_base/project/'eb_mod'/f'wy_gdg_{us_huc}'/'Hydrology'/f'{event_us}_output.dss')
        fid_us = HecDss.Open(str(dss_us))        
        hms_event_us = event_us[-10:].replace('_','-')
        # event_us = events_dict_us[us_huc][j][ri].replace('-','_')
        event_finder_us = f'[\d\w\S\s]+{j}/[\d\w\S]+{hms_event_us}[\d\w\S]+'
        #get all relevant dss paths and condense using wildcard
        r = re.compile(event_finder_us)
        dss_list_us = fid_us.getPathnameList(pathname_pattern,sort=1)
        dss_matches_us = list(filter(r.match, dss_list_us))
        assert len(dss_matches_us)>0, f'no output data found for {event_us} in {dss_us}'
        #write new dss file
        hg_name = create_bc_from_junc(bc_connections,fid,fid_us,stage_data_ds,flow_data_ds,dss_matches_us,rating_or_stage)
        bc_connections['dss_path'][us_huc][ri] = hg_name
        

In [15]:
flow_number_dict = {}
domain_name_dict = {}
for us_huc, j in  bc_connections['junctions'].items():
    us_model_name = f'wy_gdg_{us_huc}' #folder within inputs folder
    ##read plan names and get relevant plan output file to be copied to upstream hucs
    plan_number_dict = {}
    flow_number_dict[us_huc] ={}
    plan_path, in_geom_path, in_flow_path = get_current_ras_files(outputs_base/project/'eb_mod'/f'wy_gdg_{us_huc}'/f'{us_model_name}.prj')
    
    #geo hdf
    hf_geo = h5py.File(str(in_geom_path)+'.hdf','r')
    #get domain names
    domain_geo = str(list(hf_geo['Geometry']['2D Flow Areas']['Attributes'])[0][0]).strip("'b\'")
    domain_name_dict[us_huc] = domain_geo
    plan_files = glob.glob(str(plan_path.parent/'*.p*[!rj][!.hdf]'))
    for p in plan_files:
        #read old text
        with open(p, "r+") as f:
            file_contents = f.read()
        plan_title_finder= '(?<=Plan Title=)[\d\w]+(?=_output)'
        f = re.search(plan_title_finder,file_contents)
        flow_finder = '(?<=Plan Title=)[\d\w]+(?=_output)'
        u = re.search(flow_finder,file_contents)
        if f:
            f_num = re.search('.p[\d]+',p)
            plan_number_dict[f.group()] = f_num.group()
            u_num = re.search('u[\d]+',file_contents)
            flow_number_dict[us_huc][f.group()] = u_num.group()

In [16]:
for us_huc in  bc_connections['dss_path'].keys():
    flow_files = glob.glob(str(outputs_base/project/'eb_mod'/f'wy_gdg_{us_huc}'/'*.u*[!.hdf]'))
    junction = bc_connections['junctions'][us_huc]
    domain_name = domain_name_dict[us_huc]
    for ri,dss_path in bc_connections['dss_path'][us_huc].items():
        
        #########
        j_ = list(tie_in_events[us_huc].keys())        
        event = tie_in_events[us_huc][j_[0]][ri].replace('-','_')
        #########
        
        dss_us = str(outputs_base/project/'eb_mod'/f'wy_gdg_{us_huc}'/'Hydrology'/f'{event}_output.dss')
        flow_num = flow_number_dict[us_huc][event]
        in_flow_path = glob.glob(str(outputs_base/project/'eb_mod'/f'wy_gdg_{us_huc}'/f'*.{flow_num}'))[0]
        dss_path = bc_connections['dss_path'][us_huc][ri]
        dss_name = str(pl.Path(dss_us).stem)+'.dss' 
        if dss_path.split('/')[2][-3:] == '_rc':
            ##############troubleshooting
            print(dss_path)
            print(in_flow_path)
            ###############
            #assume rating curve condition
            a = update_flow_file(dss_name,dss_path,in_flow_path,huc,domain_name)
        elif dss_path.split('/')[2][-10:] == '_stage_hyd':
            #assumes ponding condition
            a = update_flow_file_stage(dss_name,dss_path,in_flow_path,huc,domain_name)
        else:
            #doesn't match any conditions
            print('issue with code assumptions, Talk to Curtis')
            break


/BCLINE/Fty Rd Crk- Grn: HUC_101_J_96_rc/-///R1_Y020_E0001_output_93/
U:\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code\outputs\wy_fy22\eb_mod\wy_gdg_1404010102\wy_gdg_1404010102.u52
/BCLINE/Fty Rd Crk- Grn: HUC_101_J_96_rc/-///R7_Y321_E0002_output_98/
U:\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code\outputs\wy_fy22\eb_mod\wy_gdg_1404010102\wy_gdg_1404010102.u53
/BCLINE/Fty Rd Crk- Grn: HUC_101_J_96_rc/-///R8_Y486_E0003_output_78/
U:\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code\outputs\wy_fy22\eb_mod\wy_gdg_1404010102\wy_gdg_1404010102.u54
/BCLINE/Fty Rd Crk- Grn: HUC_101_J_96_rc/-///R5_Y285_E0002_output_66/
U:\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code\outputs\wy_fy22\eb_

In [18]:
print('complete')

complete


## END